In [ ]:
# Count the number of prefixes that prepended path towards the scrubber and has a differnt upstream provider
# Hypothesis: if a prefix has path prepending towards scrubber, it should have other upstream without path prepending
# This method uses parallel processing

import pybgpstream
import datetime
import pandas as pd
import multiprocessing
from functools import partial
import sys
import os

asn = "32787"
year = "2024"
seen_time = 1704067200

confirmed_customers1 = pd.read_csv("/home/shyam/jupy/ddos_scrubber/data/confirmed_customers_as"+asn+"_"+year+".csv")
confirmed_customers1_unique_prefixes = confirmed_customers1['prefix'].unique()

path_prepending = pd.read_csv("/home/shyam/jupy/ddos_scrubber/data/unique_optimized_provider_as"+asn+"_path_prepend_01_jan_"+year+".csv")
path_prepending_unique_prefixes = path_prepending['prefix'].unique()


new_prefixes_prepending = list(set(path_prepending_unique_prefixes)-set(confirmed_customers1_unique_prefixes))

def suppress_c_stderr():
    """Redirect stderr to null device."""
    sys.stderr.flush()  # Flush existing errors
    stderr_fd = sys.stderr.fileno()
    with open(os.devnull, 'w') as fnull:
        os.dup2(fnull.fileno(), stderr_fd)  # Redirect stderr to null device

def find_providers_superprefix(prefix, seen_time):
    print("Check for prefix %s" %prefix)
    from_time = seen_time  # - 7200 # -7200 for 2 hours early
    until_time = seen_time  # + 7200 for 2 hours later

    # Convert to UTC time
    from_time_utc = datetime.datetime.utcfromtimestamp(from_time)
    until_time_utc = datetime.datetime.utcfromtimestamp(until_time)

    # Convert UTC datetime to string
    from_time_utc_str = from_time_utc.strftime("%Y-%m-%d %H:%M:%S")
    until_time_utc_str = until_time_utc.strftime("%Y-%m-%d %H:%M:%S")

    print("Time %s and %s" % (from_time_utc_str, until_time_utc_str))

    # Suppress C-level errors
    suppress_c_stderr()

    stream = pybgpstream.BGPStream(
        from_time=from_time_utc_str, until_time=until_time_utc_str,
        record_type="ribs",
        filter="prefix exact "+ prefix +" and path !_"+asn+"_"

    )
    p = []  # List containing immediate provider
    o = []  # Origin ASN
    prefixes = []

    print("Extracting records..")

    # Find paths from DDoS scrubber to route collectors
    for rec in stream.records():
        time = rec.time
        for elem in rec:
            pfx = elem.fields["prefix"]
            # Find second ASN in an AS path
            as_path = elem.fields["as-path"]

            #         Convert as_path into list
            as_path_list = as_path.split()
            orig = as_path_list[-1]
            # Discard different forms of origins example AS-Set, confederation set/sequence.
            # Take only a single AS origin which is common for a scrubbing activity
            if pfx != '0.0.0.0/0' and len(as_path_list) > 1:
                print(elem)
                # Find second ASN in an AS path
                second_as = as_path_list[-2]
                # Extract its upstream provider to check if that one contains an scrubbers' ASN
                p.append(second_as)  # Remove AS
                o.append(orig)
                prefixes.append(pfx)

    peers = list(set(p))

    print("List of providers is ", len(peers))
    print("List of origins is ", list(set(o)))
    print("List of prefixes are ", list(set(prefixes)))
    print("Providers ", peers)

find_providers_superprefix
# Number of processes to use (typically match the number of CPU cores)
num_processes = multiprocessing.cpu_count()

# Create a Pool of workers
with multiprocessing.Pool(num_processes) as pool:
    # Use map to distribute the work across processes and partial to add second argument
    results = pool.map(partial(find_providers_superprefix, seen_time=seen_time), new_prefixes_prepending[0:1])

In [ ]:
# Count the number of prefixes that prepended path towards the scrubber and has a differnt upstream provider
# Hypothesis: if a prefix has path prepending towards scrubber, it should have other upstream without path prepending
# This method does not use parallel processing, but simple for loop

import pybgpstream
import datetime
import pandas as pd
import multiprocessing
from functools import partial
import sys
import os

asn = "32787"
year = "2024"
seen_time = 1704067200
counter = 0 # Counter to count how many prefixes have different upstreams

confirmed_customers1 = pd.read_csv("/home/shyam/jupy/ddos_scrubber/data/confirmed_customers_as"+asn+"_"+year+".csv")
confirmed_customers1_unique_prefixes = confirmed_customers1['prefix'].unique()

path_prepending = pd.read_csv("/home/shyam/jupy/ddos_scrubber/data/unique_optimized_provider_as"+asn+"_path_prepend_01_jan_"+year+".csv")
path_prepending_unique_prefixes = path_prepending['prefix'].unique()


new_prefixes_prepending = list(set(path_prepending_unique_prefixes)-set(confirmed_customers1_unique_prefixes))

def suppress_c_stderr():
    """Redirect stderr to null device."""
    sys.stderr.flush()  # Flush existing errors
    stderr_fd = sys.stderr.fileno()
    with open(os.devnull, 'w') as fnull:
        os.dup2(fnull.fileno(), stderr_fd)  # Redirect stderr to null device

def find_providers_superprefix(prefix, seen_time):
    print("Check for prefix %s" %prefix)
    from_time = seen_time  # - 7200 # -7200 for 2 hours early
    until_time = seen_time  # + 7200 for 2 hours later
    global counter
    # Convert to UTC time
    from_time_utc = datetime.datetime.utcfromtimestamp(from_time)
    until_time_utc = datetime.datetime.utcfromtimestamp(until_time)

    # Convert UTC datetime to string
    from_time_utc_str = from_time_utc.strftime("%Y-%m-%d %H:%M:%S")
    until_time_utc_str = until_time_utc.strftime("%Y-%m-%d %H:%M:%S")

    print("Time %s and %s" % (from_time_utc_str, until_time_utc_str))

    # Suppress C-level errors
    suppress_c_stderr()

    stream = pybgpstream.BGPStream(
        from_time=from_time_utc_str, until_time=until_time_utc_str,
        record_type="ribs",
        filter="prefix exact "+ prefix +" and path !_"+asn+"_"

    )

    print("Extracting records..")

    # Find paths from DDoS scrubber to route collectors
    for rec in stream.records():
        time = rec.time
        for elem in rec:
           print(elem)
           counter +=1

    print("Counter ", counter)


for pfx in new_prefixes_prepending[0:5]:
    # Use map to distribute the work across processes and partial to add second argument
    find_providers_superprefix(pfx, seen_time)

In [7]:
# We found many prefixes that prepended their paths towards scrubbers. See their BGP routing data before and after the date
# 1. Find unique prefixes that prepend the origin ASN
import pybgpstream
import datetime
import pandas as pd
import multiprocessing
from functools import partial
import sys
import os

asn = "32787"
year = "2024"
seen_time = 1704067200

confirmed_customers1 = pd.read_csv("/home/shyam/jupy/ddos_scrubber/data/confirmed_customers_as"+asn+"_"+year+".csv")
confirmed_customers1_unique_prefixes = confirmed_customers1['prefix'].unique()

path_prepending = pd.read_csv("/home/shyam/jupy/ddos_scrubber/data/unique_optimized_provider_as"+asn+"_path_prepend_01_jan_"+year+".csv")
path_prepending_unique_prefixes = path_prepending['prefix'].unique()


new_prefixes_prepending = list(set(path_prepending_unique_prefixes)-set(confirmed_customers1_unique_prefixes))

def suppress_c_stderr():
    """Redirect stderr to null device."""
    sys.stderr.flush()  # Flush existing errors
    stderr_fd = sys.stderr.fileno()
    with open(os.devnull, 'w') as fnull:
        os.dup2(fnull.fileno(), stderr_fd)  # Redirect stderr to null device

def find_providers_superprefix(prefix, seen_time):
    print("Check for prefix %s" %prefix)
    from_time = seen_time  # - 7200 # -7200 for 2 hours early
    until_time = seen_time  # + 7200 for 2 hours later

    # Convert to UTC time
    from_time_utc = datetime.datetime.utcfromtimestamp(from_time)
    until_time_utc = datetime.datetime.utcfromtimestamp(until_time)

    # Convert UTC datetime to string
    from_time_utc_str = from_time_utc.strftime("%Y-%m-%d %H:%M:%S")
    until_time_utc_str = until_time_utc.strftime("%Y-%m-%d %H:%M:%S")

    print("Time %s and %s" % (from_time_utc_str, until_time_utc_str))

    # Suppress C-level errors
    suppress_c_stderr()

    stream = pybgpstream.BGPStream(
        from_time=from_time_utc_str, until_time=until_time_utc_str,
        record_type="ribs",
        filter="prefix exact "+ prefix +" and path !_"+asn+"_"

    )
    p = []  # List containing immediate provider
    o = []  # Origin ASN
    prefixes = []

    print("Extracting records..")

    # Find paths from DDoS scrubber to route collectors
    for rec in stream.records():
        time = rec.time
        for elem in rec:
            pfx = elem.fields["prefix"]
            # Find second ASN in an AS path
            as_path = elem.fields["as-path"]

            #         Convert as_path into list
            as_path_list = as_path.split()
            orig = as_path_list[-1]
            # Discard different forms of origins example AS-Set, confederation set/sequence.
            # Take only a single AS origin which is common for a scrubbing activity
            if pfx != '0.0.0.0/0' and len(as_path_list) > 1:
                print(elem)
                # Find second ASN in an AS path
                second_as = as_path_list[-2]
                # Extract its upstream provider to check if that one contains an scrubbers' ASN
                p.append(second_as)  # Remove AS
                o.append(orig)
                prefixes.append(pfx)

    peers = list(set(p))

    print("List of providers is ", len(peers))
    print("List of origins is ", list(set(o)))
    print("List of prefixes are ", list(set(prefixes)))
    print("Providers ", peers)

find_providers_superprefix
# Number of processes to use (typically match the number of CPU cores)
num_processes = multiprocessing.cpu_count()

# Create a Pool of workers
with multiprocessing.Pool(num_processes) as pool:
    # Use map to distribute the work across processes and partial to add second argument
    results = pool.map(partial(find_providers_superprefix, seen_time=seen_time), new_prefixes_prepending[0:1])


224 number of unique prefixes that contain AS32787 as provider after checking path prepending.


In [12]:
prefix = "192.168.1.0/24"
superprefix = "192.168.0.0/22"
def is_less_specific(superprefix, prefix):
    
    """
    Check if the given prefix is less specific than the supernet.
    """
    import ipaddress

    prefix_net = ipaddress.ip_network(prefix)
    supernet_net = ipaddress.ip_network(superprefix)
    return supernet_net.prefixlen < prefix_net.prefixlen 
#     return supernet_net.subnet_of(prefix_net) and supernet_net.prefixlen < prefix_net.prefixlen 